# Hamiltonian Time Evolution

## Approximating exponentials

In [ ]:
import pennylane as qml
from pennylane import numpy as np

### Distant electrons

In [ ]:
n_bits=2
dev = qml.device("default.qubit", wires=range(n_bits))

@qml.qnode(dev)
def two_distant_spins(B, time):
    """Circuit for evolving the state of two distant electrons in a magnetic field.
    
    Args:
        B (float): The strength of the field, assumed to point in the z direction.
        time (float): The time we evolve the electron wavefunction for.

    Returns: 
        array[complex]: The quantum state after evolution.
    """
    e = 1.6e-19
    m_e = 9.1e-31
    alpha = B*e/(2*m_e)
    ##################
    # YOUR CODE HERE #
    ##################
    qml.RZ(-2*alpha*time, wires = 0)
    qml.RZ(-2*alpha*time, wires = 1)

    return qml.state()

B, time = 1, 1
print(qml.draw(two_distant_spins)(B, time))

### Nearby electrons

In [ ]:
n_bits=2
dev = qml.device("default.qubit", wires=range(n_bits))

@qml.qnode(dev)
def two_close_spins_X(alpha, beta, time, n):
    """Circuit for evolving the state of two electrons with an X coupling.
    
    Args:
        alpha (float): The coefficient related to the strength of the field, assumed to point in the z direction.
        beta (float): The coefficient related to the strength of the coupling between electrons.
        time (float): The time we evolve the electron wavefunction for.
        n (int): The number of steps in our Trotterization.

    Returns: 
        array[complex]: The quantum state after evolution.
    """

    ##################
    # YOUR CODE HERE #
    ##################
    for _ in range(n):
        qml.IsingXX(2*beta*time/n, wires = [0,1])
        qml.RZ(2*alpha*time/n, wires = 0)
        qml.RZ(2*alpha*time/n, wires = 1)
    
    
    return qml.state()

alpha, beta, time, n = 1, 1, 1, 1

print(qml.draw(two_close_spins_X)(alpha, beta, time, n ))

### Building th eIsing Hamiltonians

In [ ]:
n_bits=2
dev = qml.device("default.qubit", wires=range(n_bits))

def ham_close_spins(alpha, J):
    """Creates the Hamiltonian for two close spins.

    Args:
        alpha (float): The coefficient related to the strength of the field, assumed to point in the z direction.
        J (list[float]): A vector of coupling coefficients [J_X, J_Y, J_Z].

    Returns:
        qml.Hamiltonian: The Hamiltonian of the system.
    """

    ##################
    # YOUR CODE HERE #
    ##################
    coeffs = [-alpha, -alpha, J[0], J[1], J[2]] # MODIFY THIS
    obs = [qml.PauliZ(0),
          qml.PauliZ(1),
          qml.PauliX(0) @ qml.PauliX(1),
          qml.PauliY(0) @ qml.PauliY(1),
          qml.PauliZ(0) @ qml.PauliZ(1),
          ] # MODIFY THIS
    return qml.dot(coeffs, obs)# Return the Hamiltonian using qml.dot

alpha, J = 1, np.array([1,1,1])
print(ham_close_spins(alpha, J))

### Trotterizing ising

In [ ]:
n_bits = 2
dev =qml.device("default.qubit", wires = n_bits)

@qml.qnode(dev)
def two_close_spins(alpha, J, time, n):
    """Circuit for evolving state of two nearby electrons with an arbitrary coupling.
    
    Args:
        alpha(float): The strength of the field, assumed to point in the z direction.
        J (array[float]): The coupling strengths J = [J_X, J_Y, J_Z] between electrons.
        time (float): The time we evolve the electron wavefunction for.
        n (int): The number of steps in our Trotterization.

    Returns: 
        array[complex]: The quantum state after evolution.
    """
    ##################
    # YOUR CODE HERE #
    ##################
    hamiltonian = ham_close_spins(alpha, J)
    qml.TrotterProduct(-hamiltonian, time, n = n)
    return qml.state()

alpha, J ,time, n= 1, np.array([1,1,1]), 1, 2
print(qml.draw(two_close_spins)(alpha, J ,time, n))